# Portfolio Construction Case Study

This notebook explores portfolio construction using solver-backed optimization and comparative method analysis.

**Key feature**: The Curriculum (Optimal) method uses `cvxpy` to solve a constrained minimum-variance optimization problem. If cvxpy unavailable, it gracefully falls back to a heuristic method.

## Workflow
1. Bootstrap imports
2. Load the case-study configuration
3. Run the solver-backed portfolio construction workflow
4. Compare with heuristic and other methods
5. Stress-test under market scenarios


In [ ]:
import sys
from pathlib import Path

def find_repo_root(start = None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'core.py').exists() and (candidate / 'configs' / 'default.yml').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root')

repo_root = find_repo_root()
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from core import build_default_constraints, construct_case_study, construct_optimal_portfolio, export_case_study, load_config, compare_strategies
from scenarios import Scenario

print(f'Repo root: {repo_root}')


## Baseline Portfolio Construction (Solver-Backed)

Run the primary case study using cvxpy for constrained minimum-variance optimization.


In [ ]:
# Run baseline case study (solver-backed by default)
print("\n" + "="*70)
print("BASELINE PORTFOLIO CONSTRUCTION")
print("="*70)
print("Using cvxpy solver for constrained minimum-variance optimization.\n")

config = load_config(repo_root / 'configs' / 'default.yml')
constraints = build_default_constraints()
result = construct_case_study(title=config.get('study_title', 'Portfolio Construction Case Study'), constraints=constraints)
paths = export_case_study(result, repo_root / config.get('output_dir', 'reports'))

print(result.to_markdown())
print('\nExported files:')
for label, path in paths.items():
    print(f'{label}: {path}')


## Strategy Comparison

Compare all three methods side by side:
- **Curriculum (Optimal)**: Solver-backed minimum-variance with cvxpy (or heuristic fallback)
- **Risk Parity**: Inverse-volatility weighted (equal risk contribution)
- **Equal Weight**: 1/N naive allocation (robustness benchmark)

Understand trade-offs, constraints, and when each method is appropriate.


In [ ]:
# Compare all three construction methods side by side
comparison = compare_strategies(
    title="Portfolio Strategy Comparison (Baseline)",
    constraints=constraints
)

print(comparison.to_markdown())


## Scenario Analysis & Stress Testing

Test portfolio robustness under market stress scenarios. Each scenario adjusts asset expectations and volatilities to simulate realistic stress regimes.


In [ ]:
# Compare strategies under RISK_OFF scenario
print("\n" + "="*70)
print("SCENARIO: RISK_OFF (Flight to Safety)")
print("="*70)
print("In this scenario: equities down, bonds rally, gold spikes.\n")

comparison_risk_off = compare_strategies(
    title="Portfolio Strategy Comparison (Risk-Off Scenario)",
    constraints=constraints,
    scenario=Scenario.RISK_OFF
)

print(comparison_risk_off.to_markdown())


In [ ]:
# Compare strategies under EQUITY_SHOCK scenario
print("\n" + "="*70)
print("SCENARIO: EQUITY_SHOCK (Severe Equity Downturn)")
print("="*70)
print("In this scenario: equities crater, REITs collapse, bonds rally sharply, gold spikes.\n")

comparison_shock = compare_strategies(
    title="Portfolio Strategy Comparison (Equity Shock Scenario)",
    constraints=constraints,
    scenario=Scenario.EQUITY_SHOCK
)

print(comparison_shock.to_markdown())


## Client Account Intake Example

Generate synthetic new-client accounts and run them through the solver-backed optimization workflow.


In [ ]:
from src import generate_client_portfolio_examples

client_examples = generate_client_portfolio_examples(count=2, seed=42, use_solver=True)
for example in client_examples:
    print("\n" + "="*70)
    print(f"CLIENT: {example.client.name} ({example.client.client_id})")
    print("="*70)
    print(f"Risk tolerance: {example.client.risk_tolerance.value}")
    print(f"Goal: {example.client.goal.value}")
    print(f"Horizon: {example.client.horizon_years} years")
    print(f"Capital: ${example.client.investable_capital:,.2f}")
    print(example.result.to_markdown())


## Key Learning Insights

### Curriculum (Optimal) - Solver-Backed Optimization
- Uses cvxpy to solve constrained minimum-variance problem (convex optimization)
- Respects explicit weight bounds defined by investment policy
- Deterministic and repeatable; finds global optimum
- Falls back to heuristic method if cvxpy unavailable
- More sophisticated than heuristics; requires accurate risk/return estimates
- Performance depends on constraint calibration

### Risk Parity (Inverse-Volatility Weighted)
- Each asset contributes equally to portfolio risk
- Simpler to maintain in dynamic markets
- Works well when volatilities are stable
- Can underperform if volatility structure changes (see stress scenarios)

### Equal Weight (1/N Naive)
- Simplest and most robust to estimation error
- Surprisingly competitive in practice
- Good benchmark for evaluating optimization quality
- Ignores market structure entirely

### Stress Testing Benefits
- Reveals how methods respond to changing market conditions
- Shows which methods provide diversification under stress
- Helps understand concentration risk and tail exposure
- Informs risk limits and hedging strategy
